# RhetFig Agreement

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import re

## Functions for reading, cleaning, and calculating

In [3]:
def return_anno(annotation,annotator):
    '''
    given an annotation result and an annotator number (int), returns the annotation created by that annotator
    '''
    for n in [0,1]:
        if annotation[n]['completed_by'] == annotator:
            return annotation[n]

In [4]:
def json_to_df(path):
    '''
    takes path to json file of annotations and returns a df 
    '''
    # load in CMT annotation json and save it to a pandas dataframe
    met_df = pd.read_json(path)
    
    # extracting labels from the annotations column
    met_df['labels'] = met_df.apply(lambda row: row.annotations[0]['result'], axis=1)
    
    # cleaning up file names 
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"^[^_]*-", '', row.file_upload), axis=1)
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"_fixed", '', row.filename[:-4].lower()), axis=1)
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"aboriginal", 'indigenous', row.filename), axis=1)
    
    # creating columns for each annotator
    anno_A = met_df.annotations[0][0]['completed_by'] #finds annotator number associated with Annotator A
    anno_B = met_df.annotations[0][1]['completed_by'] #finds annotator number associated with Annotator B
    met_df['anno_A'] = met_df.apply(lambda row: return_anno(row.annotations,anno_A)['result'], axis=1)
    met_df['anno_B'] = met_df.apply(lambda row: return_anno(row.annotations,anno_B)['result'], axis=1)

    return met_df[['anno_A', 'anno_B']]

## Interannotator Agreement

In [17]:
# set file paths (THIS IS THE ONLY THING THAT NEEDS TO CHANGE)
file_path_annotator_interanno = 'RhetFigInterAnno.json'

# from path to dataframe
annotator_interanno_df = json_to_df(file_path_annotator_interanno)

In [18]:
dict_A = {}
for comment in range(20):
    # print(comment)
    comment_dic = {}
    for x in annotator_interanno_df.anno_A[comment]:
        val = x['value']
        indices = [val['start'], val['end']]
      #  print(indices)
        labels = []
        for label in val['taxonomy']:
            labels.append(label[-1])
        if 'Sarcasm' in labels or 'Base Irony' in labels or 'Surreal Irony' in labels:
            try:
                labels.remove('Irony')
            except:
                print('no Irony')
       # print(labels)
        comment_dic[str(indices)] = labels
    dict_A[str(comment)] = comment_dic
annoA_df = pd.json_normalize(dict_A).T.reset_index()
annoA_df.columns = ['indices', 'anno_A']
annoA_df

,indices,anno_A
0,"0.[82, 103]",[Sarcasm]
1,"0.[170, 245]",[Hyperbole]
2,"0.[122, 170]",[Sarcasm]
3,"1.[386, 528]",[Antithesis]
4,"2.[0, 67]",[Rhetorical Question]
...,...,...
66,"18.[854, 881]",[Rhetorical Question]
67,"18.[1033, 1056]",[Hyperbole]
68,"19.[101, 165]",[Rhetorical Question]
69,"19.[166, 229]","[Puns/Wordplay, Allusion]"


In [19]:
dict_B = {}
for comment in range(20):
    # print(comment)
    comment_dic = {}
    for x in annotator_interanno_df.anno_B[comment]:
        val = x['value']
        # print(val['text'])
        indices = [val['start'], val['end']]
        # print(indices)
        labels = []
        for label in val['taxonomy']:
            labels.append(label[-1])
        if 'Sarcasm' in labels or 'Base Irony' in labels or 'Surreal Irony' in labels:
            try: 
                labels.remove('Irony')
            except:
                print('no irony')
        # print(labels)
        comment_dic[str(indices)] = labels
    dict_B[str(comment)] = comment_dic
annoB_df = pd.json_normalize(dict_B).T.reset_index()
annoB_df.columns = ['indices', 'anno_B']
annoB_df

no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony
no irony


,indices,anno_B
0,"0.[245, 297]",[Hyperbole]
1,"0.[83, 102]",[Sarcasm]
2,"0.[170, 245]",[Hyperbole]
3,"0.[19, 121]",[Hyperbole]
4,"1.[445, 529]",[Sarcasm]
...,...,...
101,"18.[854, 881]",[Rhetorical Question]
102,"18.[1033, 1056]",[Hyperbole]
103,"18.[523, 643]",[Hyperbole]
104,"19.[446, 486]",[Sarcasm]


In [20]:
merged_df = annoA_df.merge(annoB_df, how='outer', on='indices')

In [33]:
# merged_df.head()

In [22]:
merged_df['span_agreement'] = merged_df.apply(lambda row: 0 if row.anno_A is np.nan or row.anno_B is np.nan else 1, axis=1)

In [29]:
# int(merged_df.span_agreement.sum())

In [30]:
# merged_df.to_csv('RhetFigInterAnno.csv')

In [61]:
len(annoA_df)

71

In [62]:
len(annoB_df)

106

In [65]:
def pooled_chance_agreement(df):
    '''
    takes a dataframe with columns 'anno_A' and 'anno_B' and return the pooled probability of chance agreement 
    '''
    # lists of the domains used by each annotator 
    anno_A_domains = []
    for row in df['anno_A']:
       # print(row)
        for metaphor in row:
            tax = metaphor['value']['taxonomy']
            if (['Irony', 'Sarcasm'] in tax or ['Irony', 'Surrealist Irony'] in tax or ['Irony', 'Base Irony'] in tax) and ['Irony'] in tax:
                tax.remove(['Irony'])
            for domain in tax:
                anno_A_domains.append(domain[0])
    anno_B_domains = []
    for row in df['anno_B']:
        for metaphor in row:
            tax = metaphor['value']['taxonomy']
            if (['Irony', 'Sarcasm'] in tax or ['Irony', 'Surrealist Irony'] in tax or ['Irony', 'Base Irony'] in tax) and ['Irony'] in tax:
                tax.remove(['Irony'])
            for domain in tax:
                anno_B_domains.append(domain[0])
    
    # total number of domains used by each annotator 
    anno_A_total = len(anno_A_domains)
    anno_B_total = len(anno_B_domains)
    
    # a list of all unique domains
    all_domains = set(anno_A_domains + anno_B_domains)
    
    # dictionaries of domains, with counts set to zero
    anno_A_domains_dict = dict.fromkeys(all_domains,0)
    anno_B_domains_dict = dict.fromkeys(all_domains,0)
    
    # adding to counts for each domain
    for domain in anno_A_domains:
        anno_A_domains_dict[domain] +=1
    for domain in anno_B_domains:
        anno_B_domains_dict[domain] +=1
    
    # setting sum of probability of chance agreement to zero
    sum_p_e = 0
    for domain in all_domains:
        # probability of chance agreement for that domain
        p_e_d = (anno_A_domains_dict[domain]/anno_A_total)*(anno_B_domains_dict[domain]/anno_B_total)
        # adding to sum
        sum_p_e += p_e_d
    # pooling
    pooled_p_e = sum_p_e/len(all_domains)
    return pooled_p_e

In [66]:
pooled_chance_agreement(annotator_interanno_df)

0.018639871850881024

In [67]:
# testing primary annotators
p_o = 28/(18+24) # from calculated strict agreement 
pooled_p_e = pooled_chance_agreement(annotator_interanno_df)
print('pooled chance agreement:', pooled_p_e)
print('cohen\'s kappa for agreement testing between annotators:',(p_o-pooled_p_e)/(1-pooled_p_e))

pooled chance agreement: 0.018639871850881024
cohen's kappa for agreement testing between annotators: 0.660335361329574
